# 01 · Descarga de RESIDE (Kaggle) y manifiesto de datos

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

Descargamos **RESIDE-OTS** (im outdoor con niebla sintética y **β conocido por
imagen**) desde un espejo de Kaggle, inspeccionamos la estructura y generamos
`reside_manifest.csv`: la tabla maestra con ruta, escena, luz atmosférica y β
de cada imagen. El β es la ancla física del etiquetado (notebook 02).

| Dataset | Uso |
|---|---|
| **RESIDE-OTS** (~miles de imágenes outdoor) | train/val — etiquetas físicas |
| SOTS-OUT (test oficial) | opcional, test secundario |
| O-HAZE (real, 45 escenas) | test externo cualitativo (ver `scripts/download_ohaze.py`) |

> ⚠️ Los espejos comunitarios de Kaggle cambian de nombre y de estructura.
> Primero ejecuta la celda de **búsqueda** y elige el dataset correcto.

In [ ]:
# Setup estándar del proyecto
import random, subprocess, sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
MANIFEST_PATH = DATA_DIR / "processed" / "reside_manifest.csv"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Datos crudos en: {RAW_DIR}")

In [ ]:
# Credenciales de Kaggle: busca kaggle.json en Drive, o súbelo ahora
import shutil
from pathlib import Path

kaggle_json = None
for cand in [Path("/content/drive/MyDrive/kaggle.json"),
             Path("/content/kaggle.json"),
             Path.home() / ".kaggle" / "kaggle.json"]:
    if cand.exists():
        kaggle_json = cand
        break

if kaggle_json is None:
    from google.colab import files
    print("Sube tu kaggle.json (kaggle.com → Account → Create New API Token)")
    up = files.upload()
    kaggle_json = Path(list(up.keys())[0])

kg = Path.home() / ".kaggle"
kg.mkdir(exist_ok=True)
shutil.copy(kaggle_json, kg / "kaggle.json")
(kg / "kaggle.json").chmod(0o600)
print("Credenciales de Kaggle listas")

In [ ]:
# BÚSQUEDA: espejos disponibles de RESIDE (los slugs cambian con el tiempo)
!kaggle datasets list -s reside --sort-by votes | head -20

De la lista anterior elige un espejo que contenga **RESIDE-OTS** (o
RESIDE-Standard con carpeta OTS). Verifica en kaggle.com la descripción antes
de descargar. Luego pega el slug (formato `usuario/dataset`) en la celda
siguiente.

In [ ]:
# DESCARGA: pega aquí el slug verificado
DATASET_SLUG = ""   # <-- POR EJEMPLO: "usuario/reside-standard" (verificar en la búsqueda)

if not DATASET_SLUG:
    print("⚠️ DATASET_SLUG vacío.")
    print("   1) revisa el resultado de la celda de búsqueda")
    print("   2) verifica el dataset en kaggle.com (que incluya OTS)")
    print("   3) escribe el slug aquí y vuelve a ejecutar esta celda")
else:
    print(f"Descargando {DATASET_SLUG} (puede tardar varios minutos)...")
    r = subprocess.run(["kaggle", "datasets", "download", "-d", DATASET_SLUG,
                        "-p", str(RAW_DIR), "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:] if r.stdout else "")
    print(r.stderr[-2000:] if r.returncode else "Descarga completada")

In [ ]:
# Inspeccionamos la estructura descargada (carpetas y tamaños)
def show_tree(root, depth=2, prefix=""):
    root = Path(root)
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + p.name +
              ("/" if p.is_dir() else ""))
        if p.is_dir() and depth > 1:
            show_tree(p, depth - 1, prefix + ("    " if last else "│   "))

show_tree(RAW_DIR, depth=2)

In [ ]:
# Detección automática de imágenes RESIDE con niebla (patrón escena_A_beta)
FILENAME_RE = re.compile(r"^(?P<scene>\d+)_(?P<atmlight>[\d.]+)_(?P<beta>[\d.]+)$")

def find_reside_hazy(root, path_filter=""):
    # Encuentra imágenes cuyo nombre sigue el patrón escena_A_beta.
    # path_filter: substring obligatorio en la ruta (p.ej. 'OTS' para solo outdoor).
    hits = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in Path(root).rglob(ext):
            if FILENAME_RE.match(p.stem) and (not path_filter or path_filter.lower() in str(p).lower()):
                hits.append(p)
    return sorted(set(hits))

# Si el espejo mezcla indoor (ITS) y outdoor (OTS), filtra por 'OTS'.
PATH_FILTER = ""      # <-- cambia a "OTS" si es necesario
hazy_paths = find_reside_hazy(RAW_DIR, path_filter=PATH_FILTER)
print(f"Imágenes con niebla detectadas: {len(hazy_paths)}")
if hazy_paths:
    print("Ejemplos:", [p.name for p in hazy_paths[:5]])

In [ ]:
# Construcción del MANIFIESTO: ruta + escena + A + beta por imagen
def parse_stem(stem):
    m = FILENAME_RE.match(stem)
    if m is None:
        return None
    return {"scene": m.group("scene"),
            "A": float(m.group("atmlight")),
            "beta": float(m.group("beta"))}

records = []
for p in hazy_paths:
    info = parse_stem(p.stem)
    if info:
        records.append({"image": p.name, "path": str(p),
                        "scene": info["scene"], "A": info["A"],
                        "beta": info["beta"], "source": "reside"})

manifest = pd.DataFrame(records)
if len(manifest):
    print(f"Manifiesto: {len(manifest)} imágenes · {manifest['scene'].nunique()} escenas")
    print(f"Rango de beta: [{manifest['beta'].min():.4f}, {manifest['beta'].max():.4f}]")
    display(manifest.head())
else:
    print("⚠️ Sin imágenes RESIDE. Usa el modo DEMO de la celda siguiente.")

## Modo DEMO (solo si la descarga falló)

Para que el pipeline completo funcione aunque Kaggle falle (p. ej. durante la
presentación), este modo sintetiza niebla **con β conocido** sobre fotos
propias (sube ~20 fotos exteriores a `Drive/camanchaca/data/raw/demo_clear/`)
usando el modelo atmosférico de dispersión con profundidad plana aproximada.

> ⚠️ Las etiquetas del modo demo son válidas **por construcción** (nosotros
> elegimos β), pero las imágenes NO son representativas: sirve para probar el
> pipeline, no para reportar resultados.

In [ ]:
# MODO DEMO: síntesis ASM con beta controlado (profundidad plana aproximada)
import numpy as np
from PIL import Image

DEMO_CLEAR = RAW_DIR / "demo_clear"
DEMO_HAZY = RAW_DIR / "demo_hazy"
DEMO_CLEAR.mkdir(parents=True, exist_ok=True)
DEMO_HAZY.mkdir(parents=True, exist_ok=True)

clear_imgs = sorted([p for p in DEMO_CLEAR.glob("*.*")
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

if len(manifest) == 0 and len(clear_imgs) > 0:
    rng = np.random.RandomState(SEED)
    betas = [0.02, 0.04, 0.08, 0.12, 0.2, 0.35]   # V = 3.912/beta: 196..11 m
    records = []
    for p in clear_imgs:
        img = np.asarray(Image.open(p).convert("RGB"), dtype=np.float64) / 255.0
        h, w = img.shape[:2]
        # profundidad relativa: de ~30 m (abajo/primero plano) a ~150 m (horizonte)
        depth = np.linspace(0.2, 1.0, h)[:, None] * np.ones((1, w)) * 150.0
        for k, beta in enumerate(rng.permutation(betas)[:3]):
            A = 0.85
            t = np.exp(-beta * depth)[..., None]
            hazy = np.clip(img * t + A * (1 - t), 0, 1)
            out = DEMO_HAZY / f"{p.stem}_{k:02d}_{A:.2f}_{beta:.3f}.jpg"
            Image.fromarray((hazy * 255).astype(np.uint8)).save(out, quality=92)
            records.append({"image": out.name, "path": str(out),
                            "scene": p.stem, "A": A, "beta": beta,
                            "source": "demo"})
    manifest = pd.DataFrame(records)
    print(f"DEMO: {len(manifest)} imágenes sintéticas con beta conocido")
elif len(manifest) == 0:
    print("Sin datos RESIDE ni fotos demo: sube fotos a", DEMO_CLEAR)
else:
    print("Manifiesto RESIDE disponible: modo demo omitido")

In [ ]:
# Subset controlado (Colab free): máximo ~12000 imágenes, proporcional por escena
MAX_IMAGES = 12000

if len(manifest) > MAX_IMAGES:
    frac = MAX_IMAGES / len(manifest)
    manifest = (manifest.groupby("scene", group_keys=False)
                        .sample(frac=frac, random_state=SEED)
                .reset_index(drop=True))
    print(f"Subset: {len(manifest)} imágenes ({manifest['scene'].nunique()} escenas)")
else:
    print(f"Se usan las {len(manifest)} imágenes disponibles")

# Verificación de legibilidad de una muestra
from PIL import Image
if len(manifest):
    for p in manifest["path"].sample(min(5, len(manifest)), random_state=SEED):
        im = Image.open(p); im.verify()
    print("Muestra de imágenes legible ✅")
else:
    print("⚠️ Manifiesto vacío: completa la descarga o el modo DEMO antes de continuar.")

In [ ]:
# Guardamos el manifiesto (Drive + local) — lo consume el notebook 02
import matplotlib.pyplot as plt

manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Manifiesto guardado: {MANIFEST_PATH} ({len(manifest)} filas)")
if len(manifest):
    manifest["beta"].hist(bins=50, figsize=(7, 3))
    plt.title("Distribución de beta en el manifiesto")
    plt.xlabel("beta [1/m]"); plt.ylabel("imágenes"); plt.show()
else:
    print("⚠️ Manifiesto vacío: revisa la descarga (slug de Kaggle) o el modo DEMO.")

## Siguiente paso

Con el manifiesto listo (`data/processed/reside_manifest.csv`), abre el
**notebook 02 · labeling_koschmieder** para convertir β en **visibilidad en
metros** (ley de Koschmieder) y asignar bandas de seguridad.

**O-HAZE (test externo real):** no viene en Kaggle. Solicítalo con correo
institucional en la página del NTIRE 2018 Image Dehazing Challenge y
organízalo con `python scripts/download_ohaze.py --zip O-HAZE.zip` (nota al
pie del notebook 06).